# Agent Target Smoke Tests

This notebook is for checking whether candidate agent targets are functional before we run adversarial attacks. The key principle is: **do not attack an agent before we know it can complete benign tasks**.

Current local status:

| Target | What it is | Current repo status |
|---|---|---|
| `toy-agent` | Our minimal tool-using agent scaffold, useful for controlled experiments and metric development. | Runnable now. Supports scripted policies and Qwen/OpenAI-compatible APIs. |
| `agentdojo` | A mature benchmark platform for prompt injection attacks and defenses in tool-using agents. | Placeholder only in this repo. Needs an adapter. |
| `injectagent` | A benchmark/scaffold for indirect prompt injection in tool-integrated LLM agents. | Placeholder only in this repo. Needs an adapter. |
| `mini-swe-agent` | A compact coding agent for software tasks. Good for hard-label pass/fail experiments. | Placeholder only in this repo. Needs an adapter. |

So today, the only fully runnable target inside this repo is `toy-agent`. The other three are the mature targets we should connect next.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "pyproject.toml").exists():
    # Useful when VS Code opens the notebook from notebooks/.
    REPO = Path.cwd().parent

assert (REPO / "pyproject.toml").exists(), f"Could not find repo root from {Path.cwd()}"
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

def run_cli(args: list[str], timeout: int = 120) -> dict:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC)
    completed = subprocess.run(
        [sys.executable, "-m", "agent_blackbox_lab", *args],
        cwd=REPO,
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if completed.returncode != 0:
        return {
            "ok": False,
            "returncode": completed.returncode,
            "stdout": completed.stdout,
            "stderr": completed.stderr,
        }
    try:
        payload = json.loads(completed.stdout)
    except json.JSONDecodeError:
        payload = completed.stdout
    return {"ok": True, "payload": payload}

REPO

## 1. What Are The Three Mature Agent Targets?

These three are not equivalent. They represent different kinds of agent systems:

### AgentDojo

AgentDojo is best understood as an **agent security benchmark platform**. It provides tool-using task environments, indirect prompt injection attacks, defenses, and evaluation logic. It is the closest match to our layered security question: where does an injected objective get stopped in an agent pipeline?

Use it for: prompt injection, tool-output contamination, defense ablation, task-utility/security tradeoff.

### InjecAgent

InjecAgent is a **benchmark/scaffold for indirect prompt injection in tool-integrated agents**. It focuses on cases where tools or external data carry adversarial instructions. It is useful for comparing ReAct-style and function-calling-style agents.

Use it for: indirect prompt injection, data exfiltration, direct harm, tool-call vulnerability analysis.

### mini-SWE-agent

mini-SWE-agent is a **compact coding agent**. It is not primarily a security benchmark, but it is useful because coding tasks naturally produce hard labels: test pass/fail, patch accepted/rejected, unsafe shell action proposed/executed.

Use it for: hard-label black-box attacks against coding agents and transfer from toy/security benchmarks to a more realistic software-agent loop.

## 2. Current Runnable Status In This Repo

This checks what the repo can run **today**. The mature targets are intentionally placeholders until we write adapters.

In [ ]:
from agent_blackbox_lab.targets import build_target
from agent_blackbox_lab.models import build_model

target_names = ["toy-agent", "agentdojo", "injectagent", "mini-swe-agent"]
target_status = []
for name in target_names:
    try:
        target = build_target(name, model=build_model("scripted-safe"), guardrail=None, max_steps=4)
        status = "runnable" if name == "toy-agent" else "placeholder_adapter_needed"
        target_status.append({"target": name, "status": status, "class": type(target).__name__})
    except Exception as exc:
        target_status.append({"target": name, "status": "error", "error": str(exc)})

target_status

## 3. Smoke Test The Current Toy Agent

This is the basic functionality gate. It should pass before we run attacks.

In [ ]:
scripted_smoke = run_cli(["smoke", "--target", "toy-agent", "--model", "scripted-safe"])
scripted_smoke

## 4. Smoke Test Toy Agent With Qwen API

This uses `api_keys` in the repo root. The file is ignored by Git. The notebook does not print the key.

In [ ]:
key_file = REPO / "api_keys"
if key_file.exists():
    qwen_smoke = run_cli([
        "smoke",
        "--target", "toy-agent",
        "--model", "openai-compatible",
        "--provider", "qwen",
        "--key-file", str(key_file),
    ], timeout=180)
else:
    qwen_smoke = {"ok": False, "reason": f"No key file found at {key_file}"}

qwen_smoke

## 5. Confirm Mature Targets Are Not Yet Wired

These commands should currently fail with a clear placeholder message. That is expected. Once we implement each adapter, these cells should become passing smoke tests.

In [ ]:
placeholder_results = {}
for target in ["agentdojo", "injectagent", "mini-swe-agent"]:
    placeholder_results[target] = run_cli([
        "smoke",
        "--target", target,
        "--model", "scripted-safe",
    ])

placeholder_results

## 6. Optional: Check Whether External Packages Are Installed

This does not install anything. It only checks the current Python environment.

In [ ]:
import importlib.util

package_checks = {
    "agentdojo": importlib.util.find_spec("agentdojo") is not None,
    "mini_swe_agent": importlib.util.find_spec("mini_swe_agent") is not None,
    # InjecAgent may not expose a stable pip import name; it is often used from a cloned repo.
    "injectagent_import_unknown": None,
}
package_checks

## 7. Readiness Matrix

Use this as our immediate checklist:

| Target | Functional smoke test | Attack test | What is missing |
|---|---:|---:|---|
| toy-agent + scripted | Yes | Yes | It is not mature; only a controlled scaffold. |
| toy-agent + Qwen | Yes, if the Qwen cell passes | Not yet systematically | Need run attack command and inspect trajectories. |
| AgentDojo | No | No | Install dependency and implement adapter in `targets.py`. |
| InjecAgent | No | No | Clone/install repo and implement adapter in `targets.py`. |
| mini-SWE-agent | No | No | Install dependency and implement coding-task adapter in `targets.py`. |

The next engineering step is to implement one mature adapter at a time, starting with AgentDojo.